# Routing, fallback y presupuesto: el modelo adecuado para cada tarea

**Facsímil 6 · Construir y operar** — capítulo 5 (routing, fallback y presupuestos por tarea).

Usar el modelo más potente para todo es cómodo y **ruinoso**. La mayoría de las tareas no necesitan el
cañón: una pregunta fácil la resuelve igual de bien un modelo barato que cuesta quince veces menos. En
este cuaderno montas un **router** que manda las tareas fáciles a un modelo barato y solo escala al caro
cuando hace falta, con un **plan B** (fallback) si el barato no está seguro, y un **tope de presupuesto**
por tarea. Compararás coste y calidad frente a la estrategia ingenua de «caro para todo», y verás que la
diferencia entre un sistema viable y uno que quema dinero suele estar aquí.

### Qué vas a aprender
- Por qué «el modelo más potente para todo» es una mala estrategia económica.
- Cómo un **router** decide a qué modelo va cada tarea.
- Qué es un **fallback** y por qué debe dispararse por una **señal observable** (la confianza), nunca
  por «saber» si el modelo acertó (que en la realidad no lo sabes).
- Cómo un **presupuesto por tarea** evita que un caso testarudo se lleve el gasto de cien.

### Cuánto cuesta
Unos 12 minutos. CPU, sin claves (se simulan los modelos).


> **Inteligencia artificial para gente curiosa** · facsímil interactivo
> 
> Web del facsímil: https://www.iaparagentecuriosa.686f6c61.dev/ · Autor: @686f6c61 · Fecha: 2026-06-26 · Versión 1.0
> 
> Este cuaderno acompaña al facsímil: ejecútalo de arriba abajo, lee cada celda de texto
> antes de correr la de código y detente en las salidas. La gracia no es que «salga», sino
> entender *por qué* sale.

In [1]:
import numpy as np
np.random.seed(1)

# Dos modelos simulados: el barato es rapido y suele acertar en lo facil; el caro
# acierta casi siempre pero cuesta ~15x mas. 'cf' = confianza (lo unico observable).
MODELOS = {
    "barato": {"coste": 0.2/1000, "ac_facil": 0.97, "ac_dificil": 0.55, "cf_facil": 0.90, "cf_dificil": 0.45},
    "caro":   {"coste": 3.0/1000, "ac_facil": 0.99, "ac_dificil": 0.92, "cf_facil": 0.95, "cf_dificil": 0.85},
}
def responde(modelo, dificil):
    m = MODELOS[modelo]
    ok = np.random.random() < (m["ac_dificil"] if dificil else m["ac_facil"])
    conf = np.clip((m["cf_dificil"] if dificil else m["cf_facil"]) + np.random.normal(0, 0.12), 0, 1)
    return ok, conf, m["coste"]

tareas = np.random.random(2000) < 0.30   # True = dificil; 30% lo son
print(f"{len(tareas)} tareas, {tareas.sum()} dificiles ({100*tareas.mean():.0f}%).")
def evalua(estrategia):
    aciertos = costes = 0
    for d in tareas:
        ok, c = estrategia(d); aciertos += ok; costes += c
    return aciertos/len(tareas), costes


2000 tareas, 588 dificiles (29%).


## 1. La línea base: el caro para todo

Funciona, acierta mucho... y paga de más en cada tarea fácil, que son la mayoría. Es la estrategia
ingenua contra la que compararemos todo lo demás.


In [2]:
def todo_caro(dificil):
    ok, conf, c = responde("caro", dificil)
    return ok, c
ac, co = evalua(todo_caro)
print(f"Caro para todo:   acierto {100*ac:.1f}%   coste ${co:.4f}")


Caro para todo:   acierto 96.7%   coste $6.0000


## 2. El router: barato por defecto, caro cuando hace falta

Un clasificador (aquí, una pista de dificultad con algo de ruido; en real, un modelo pequeño o reglas)
decide a dónde va cada tarea. Si parece fácil, al barato; si parece difícil, al caro. Y —esto es lo
importante— si el barato responde con **poca confianza**, escalamos al caro. Fíjate en que el fallback
**no** mira si el barato acertó (eso no lo sabes en producción): mira una señal observable, la
confianza. Decidir el fallback por un oráculo de acierto sería hacer trampa.


In [3]:
def router(dificil):
    coste = 0.0
    parece_dificil = dificil if np.random.random() < 0.85 else (not dificil)  # clasificador con ruido
    modelo = "caro" if parece_dificil else "barato"
    ok, conf, c = responde(modelo, dificil); coste += c
    if modelo == "barato" and conf < 0.7:        # fallback por BAJA CONFIANZA
        ok, conf, c = responde("caro", dificil); coste += c
    return ok, coste

ac_r, co_r = evalua(router)
print(f"Caro para todo:      acierto {100*ac:.1f}%   coste ${co:.4f}")
print(f"Router + fallback:   acierto {100*ac_r:.1f}%   coste ${co_r:.4f}")
print(f"\nAhorro: {100*(1-co_r/co):.0f}% del coste, con una diferencia de acierto de {100*(ac-ac_r):+.1f} puntos.")


Caro para todo:      acierto 96.7%   coste $6.0000
Router + fallback:   acierto 96.2%   coste $2.8530

Ahorro: 52% del coste, con una diferencia de acierto de +0.5 puntos.


**Ese es el trato del routing.** Recortas una buena parte de la factura sacrificando muy poca calidad,
porque la mayoría de las tareas no necesitaban el modelo caro. El fallback por confianza te cubre las
veces que el barato se queda corto. No es magia: es no usar un camión para llevar una carta.


## 3. El tope de presupuesto: que una tarea no se desboque

Un fallback mal puesto puede encadenar reintentos carísimos. Un **presupuesto por tarea** corta: si el
coste acumulado supera un límite, se para y se devuelve lo mejor que se tenga. Es una red de seguridad
económica, igual que el tope de pasos lo era para un agente.


In [4]:
PRESUPUESTO = 4.0/1000
def router_con_tope(dificil):
    coste = 0.0; ok = False
    for modelo in ["barato", "caro"]:            # escala mientras dude y haya presupuesto
        ok, conf, c = responde(modelo, dificil); coste += c
        if conf >= 0.7 or coste >= PRESUPUESTO:
            break
    return ok, coste
ac_t, co_t = evalua(router_con_tope)
print(f"Router con tope de presupuesto: acierto {100*ac_t:.1f}%   coste ${co_t:.4f}")
print("El tope evita que una tarea testaruda (reintentos) se lleve el presupuesto de cien.")


Router con tope de presupuesto: acierto 94.8%   coste $2.3020
El tope evita que una tarea testaruda (reintentos) se lleve el presupuesto de cien.


## 4. Experimento: ¿cuándo deja de compensar el router?

El routing brilla cuando la mayoría de las tareas son fáciles. Si casi todas son difíciles, casi todo
acaba yendo al caro y el ahorro se evapora. Lo medimos variando la proporción de tareas difíciles.


In [5]:
def router_simple(dificil):
    coste = 0.0
    parece = dificil if np.random.random() < 0.85 else (not dificil)
    modelo = "caro" if parece else "barato"
    ok, conf, c = responde(modelo, dificil); coste += c
    if modelo == "barato" and conf < 0.7:
        ok, conf, c = responde("caro", dificil); coste += c
    return ok, coste

print("% tareas dificiles | coste 'caro para todo' | coste router | ahorro")
base = np.random.random(2000)
for frac in [0.1, 0.3, 0.6, 0.9]:
    tareas = base < frac
    co_caro = sum(responde("caro", d)[2] for d in tareas)
    co_rout = sum(router_simple(d) for d in tareas) if False else None  # evita doble muestreo
    co_rout = 0.0
    for d in tareas: co_rout += router_simple(d)[1]
    print(f"        {frac*100:>2.0f}%         |        ${co_caro:.4f}        |   ${co_rout:.4f}  |  {100*(1-co_rout/co_caro):.0f}%")
print("\nCuanto mas faciles son las tareas, mas ahorra el router. Si casi todas son dificiles, poco margen.")


% tareas dificiles | coste 'caro para todo' | coste router | ahorro
        10%         |        $6.0000        |   $1.9230  |  68%
        30%         |        $6.0000        |   $2.8948  |  52%
        60%         |        $6.0000        |   $4.2496  |  29%
        90%         |        $6.0000        |   $5.5346  |  8%

Cuanto mas faciles son las tareas, mas ahorra el router. Si casi todas son dificiles, poco margen.


## 5. Pruébalo tú

1. **Empeora el clasificador** (baja `0.85` a `0.6`): si no sabes distinguir fácil de difícil, el routing
   pierde sentido. La calidad del router depende de la calidad de esa decisión.
2. **Sube el coste del caro** a 30×. ¿Cuánto más agresivo conviene ser enviando al barato?
3. **Cambia el umbral de confianza** del fallback (0.7) a 0.5 y a 0.9. ¿Cómo afecta al equilibrio
   coste/calidad? Un umbral alto escala más (más caro, más calidad).
4. **Tres niveles:** añade un modelo «medio» entre el barato y el caro. ¿Mejora el equilibrio?


## 6. Errores comunes

- **Mandarlo todo al modelo caro.** Cómodo y ruinoso. La mayoría de tareas no lo necesitan.
- **Disparar el fallback por un oráculo de acierto.** En producción no sabes si el modelo acertó; el
  fallback debe usar una señal observable (confianza, validación, error explícito).
- **No poner presupuesto por tarea.** Los reintentos pueden descontrolarse. El tope es la red de
  seguridad económica.
- **Olvidar que el router vale lo que vale el clasificador.** Si no distingues fácil de difícil, no hay
  routing que te salve.


## 7. Qué te llevas

- **Enrutar** (barato por defecto, caro cuando hace falta) recorta gran parte de la factura perdiendo muy
  poca calidad: la mayoría de tareas no necesitan el modelo más potente.
- El **fallback** debe decidirse por una **señal observable** (confianza), no por saber el resultado; el
  **presupuesto por tarea** evita que los reintentos se descontrolen.
- La pieza crítica es el **clasificador de dificultad**: el routing vale lo que vale esa decisión.

**Para seguir:** el capítulo 7 lleva esto a despliegues progresivos (canary, rollback); el facsímil 7
mide si la calidad que sacrificas con el modelo barato es asumible para tu caso.


---

### Ficha del cuaderno

- **Obra:** *Inteligencia artificial para gente curiosa* (facsímil interactivo).
- **Web:** https://www.iaparagentecuriosa.686f6c61.dev/
- **Autor:** @686f6c61
- **Fecha:** 2026-06-26
- **Versión:** 1.0

*Material pedagógico. Las salidas que ves son reales: se generan al ejecutar el código, no están escritas a mano. Si cambias algo, cambiarán: esa es la idea.*